# Phase 3 step 1 — Save raw baseline predictions (no training)

Runs the Phase 2 model (`baseline_r50/model_final.pth` on Drive) once on the 141 validation and 180 test
images and saves every mask with score >= 0.2, **before** any post-processing. All Phase 3 post-processing
experiments (score threshold, closing kernel, disk filter, shape filter) and new submission CSVs are then
made on the local computer, without Colab.

Time on a T4: install 5-8 min + predictions ~5 min.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`; Colab secret `GITHUB_TOKEN` as before.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
OUT_DIR = f"{DRIVE_DIR}/output"            # Phase 2 checkpoint lives here
PHASE3_DIR = f"{DRIVE_DIR}/Phase 3"         # results of this notebook
os.makedirs(PHASE3_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
WEIGHTS = f"{OUT_DIR}/detectron2/baseline_r50/model_final.pth"
print("weights found:", os.path.exists(WEIGHTS))

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5-8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.is_available())

## 4. Save raw predictions (~5 min)
Writes `raw_preds_baseline_r50_val.json.gz` and `raw_preds_baseline_r50_test.json.gz`, then copies them to
`MyDrive/solar_filament/Phase 3/`.

In [ ]:
!python run_pipeline.py --step dump --name baseline_r50 --weights "{WEIGHTS}"
import shutil
for split in ("val", "test"):
    src = f"{OUT_DIR}/raw_preds_baseline_r50_{split}.json.gz"
    shutil.copy(src, PHASE3_DIR)
    print(split, os.path.getsize(src) // 1024, "KB ->", PHASE3_DIR)

## 5. Send results back
Download the two files (also in `MyDrive/solar_filament/Phase 3/`) and put them in the local folder
`Result Phase 3/`, then tell Claude.

In [ ]:
from google.colab import files
for split in ("val", "test"):
    files.download(f"{PHASE3_DIR}/raw_preds_baseline_r50_{split}.json.gz")